# Task 4 — Improvement Cycle

Starting from the baseline scores in Task 3, we run experiments to improve description quality. Each experiment documents what changed, why, and the resulting scores.

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import (
    DATASET_PATH,
    GENERATION_MODEL,
    NEBIUS_BASE_URL,
    OUTPUT_EXCEL,
)
from shared.constants import CRITERIA, SYSTEM_PROMPT, SYSTEM_PROMPT_V2, final_score
from shared.utils import build_user_message, calculate_cost

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

df_products = pd.read_excel(DATASET_PATH)
df_baseline = pd.read_excel(OUTPUT_EXCEL)
print(f"{len(df_products)} products, baseline has {len(df_baseline)} rows")

50 products, baseline has 50 rows


In [2]:
def run_experiment(prompt: str, model: str = GENERATION_MODEL, **gen_kwargs) -> pd.DataFrame:
    """Generate descriptions for all products and return a results DataFrame."""
    defaults = {"temperature": 0.7, "max_tokens": 200}
    defaults.update(gen_kwargs)

    results = []
    for idx, row in df_products.iterrows():
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": build_user_message(row)},
        ]
        start = time.perf_counter()
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            **defaults,
        )
        elapsed_ms = (time.perf_counter() - start) * 1000
        usage = response.usage
        desc = response.choices[0].message.content.strip()
        results.append({
            "product_name": row["product_name"],
            "generated_description": desc,
            "word_count": len(desc.split()),
            "latency_ms": round(elapsed_ms, 1),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens,
        })
        print(f"  [{idx + 1}/{len(df_products)}] {row['product_name']}: {len(desc.split())} words, {elapsed_ms:.0f}ms")

    return pd.DataFrame(results)


def summarize(results_df: pd.DataFrame):
    """Print quick stats for an experiment run."""
    wc = results_df["word_count"]
    in_range = wc.between(50, 90).sum()
    print(f"Word count: mean={wc.mean():.1f}, min={wc.min()}, max={wc.max()}")
    print(f"In 50-90 range: {in_range}/{len(results_df)} ({in_range/len(results_df):.0%})")
    print(f"Latency: mean={results_df['latency_ms'].mean():.0f}ms")

---
## Experiment 1: Improved Prompt with Few-Shot Example

**What changed:** Added a concrete example of a good description and tightened the word-count instruction.

**Why:** The baseline prompt may be too vague for a 9B model. A few-shot example gives a concrete target for length and tone.

In [3]:
print("Running experiment 1...")
exp1_results = run_experiment(SYSTEM_PROMPT_V2)
print()
summarize(exp1_results)

Running experiment 1...
  [1/50] Apple iPhone 15 Pro: 62 words, 1309ms
  [2/50] Samsung Galaxy S24 Ultra: 81 words, 1040ms
  [3/50] Google Pixel 8 Pro: 83 words, 1006ms
  [4/50] Sony WH‑1000XM5 Headphones: 74 words, 1022ms
  [5/50] Bose QuietComfort Ultra Earbuds: 55 words, 821ms
  [6/50] Amazon Echo Dot (5th Gen): 79 words, 1226ms
  [7/50] Dell XPS 13 9310 Laptop: 82 words, 1126ms
  [8/50] Apple MacBook Air 13″ (M3): 59 words, 850ms
  [9/50] Microsoft Surface Pro 10: 83 words, 1076ms
  [10/50] Garmin Forerunner 255: 70 words, 1247ms
  [11/50] Fitbit Charge 6: 74 words, 920ms
  [12/50] GoPro HERO12 Black: 71 words, 1006ms
  [13/50] DJI Mini 4 Pro Drone: 55 words, 938ms
  [14/50] Nintendo Switch OLED: 79 words, 921ms
  [15/50] PlayStation 5 Slim: 77 words, 1023ms
  [16/50] Xbox Series X: 78 words, 1127ms
  [17/50] Instant Pot Duo 6‑Quart: 61 words, 1126ms
  [18/50] Keurig K‑Supreme Plus Smart: 69 words, 937ms
  [19/50] Vitamix 5200 Blender: 60 words, 1521ms
  [20/50] Dyson V15 Detect Va

---
## Experiment 2: Lower Temperature

**What changed:** Reduced temperature from 0.7 to 0.3.

**Why:** Lower temperature reduces randomness, which should improve consistency in following the length constraint and reduce hallucination (better Grounding).

In [4]:
print("Running experiment 2...")
exp2_results = run_experiment(SYSTEM_PROMPT_V2, temperature=0.3)
print()
summarize(exp2_results)

Running experiment 2...
  [1/50] Apple iPhone 15 Pro: 75 words, 1045ms
  [2/50] Samsung Galaxy S24 Ultra: 99 words, 1208ms
  [3/50] Google Pixel 8 Pro: 75 words, 932ms
  [4/50] Sony WH‑1000XM5 Headphones: 72 words, 1091ms
  [5/50] Bose QuietComfort Ultra Earbuds: 58 words, 935ms
  [6/50] Amazon Echo Dot (5th Gen): 70 words, 918ms
  [7/50] Dell XPS 13 9310 Laptop: 74 words, 1126ms
  [8/50] Apple MacBook Air 13″ (M3): 89 words, 1228ms
  [9/50] Microsoft Surface Pro 10: 72 words, 1024ms
  [10/50] Garmin Forerunner 255: 73 words, 1331ms
  [11/50] Fitbit Charge 6: 92 words, 1229ms
  [12/50] GoPro HERO12 Black: 79 words, 1075ms
  [13/50] DJI Mini 4 Pro Drone: 66 words, 941ms
  [14/50] Nintendo Switch OLED: 64 words, 906ms
  [15/50] PlayStation 5 Slim: 72 words, 925ms
  [16/50] Xbox Series X: 79 words, 1069ms
  [17/50] Instant Pot Duo 6‑Quart: 71 words, 943ms
  [18/50] Keurig K‑Supreme Plus Smart: 78 words, 995ms
  [19/50] Vitamix 5200 Blender: 51 words, 823ms
  [20/50] Dyson V15 Detect Vacuu

---
## Experiment 3: Different Model

**What changed:** Switched to Meta-Llama-3.1-8B-Instruct (the other available model).

**Why:** Different architectures have different strengths. Llama may follow length constraints better or produce more natural phrasing.

In [5]:
from shared.config import JUDGE_MODEL

print(f"Running experiment 3 with {JUDGE_MODEL}...")
exp3_results = run_experiment(SYSTEM_PROMPT_V2, model=JUDGE_MODEL, temperature=0.3)
print()
summarize(exp3_results)

Running experiment 3 with meta-llama/Meta-Llama-3.1-8B-Instruct...
  [1/50] Apple iPhone 15 Pro: 66 words, 1865ms
  [2/50] Samsung Galaxy S24 Ultra: 76 words, 3301ms
  [3/50] Google Pixel 8 Pro: 76 words, 2887ms
  [4/50] Sony WH‑1000XM5 Headphones: 63 words, 1571ms
  [5/50] Bose QuietComfort Ultra Earbuds: 62 words, 1539ms
  [6/50] Amazon Echo Dot (5th Gen): 67 words, 2659ms
  [7/50] Dell XPS 13 9310 Laptop: 70 words, 2144ms
  [8/50] Apple MacBook Air 13″ (M3): 68 words, 1533ms
  [9/50] Microsoft Surface Pro 10: 62 words, 2157ms
  [10/50] Garmin Forerunner 255: 77 words, 1708ms
  [11/50] Fitbit Charge 6: 71 words, 1958ms
  [12/50] GoPro HERO12 Black: 62 words, 2785ms
  [13/50] DJI Mini 4 Pro Drone: 70 words, 1946ms
  [14/50] Nintendo Switch OLED: 65 words, 2337ms
  [15/50] PlayStation 5 Slim: 64 words, 2548ms
  [16/50] Xbox Series X: 74 words, 2898ms
  [17/50] Instant Pot Duo 6‑Quart: 61 words, 2922ms
  [18/50] Keurig K‑Supreme Plus Smart: 59 words, 2295ms
  [19/50] Vitamix 5200 Blende

---
## Compare Experiments

Pick the best run based on the proportion of descriptions in the 50–90 word range and overall quality.

In [6]:
comparison = pd.DataFrame({
    "experiment": ["Baseline", "Exp1: Few-shot prompt", "Exp2: Low temp", "Exp3: Llama model"],
    "in_range_pct": [
        df_baseline["generated_description"].apply(lambda x: 50 <= len(str(x).split()) <= 90).mean(),
        exp1_results["word_count"].between(50, 90).mean(),
        exp2_results["word_count"].between(50, 90).mean(),
        exp3_results["word_count"].between(50, 90).mean(),
    ],
    "mean_latency_ms": [
        df_baseline["latency_ms"].mean(),
        exp1_results["latency_ms"].mean(),
        exp2_results["latency_ms"].mean(),
        exp3_results["latency_ms"].mean(),
    ],
})
comparison["in_range_pct"] = (comparison["in_range_pct"] * 100).round(1)
comparison["mean_latency_ms"] = comparison["mean_latency_ms"].round(0)
comparison

,experiment,in_range_pct,mean_latency_ms
0,Baseline,98.0,1043.0
1,Exp1: Few-shot prompt,100.0,986.0
2,Exp2: Low temp,96.0,994.0
3,Exp3: Llama model,100.0,3113.0


## Manual Re-evaluation of Best Experiment

Re-rate a sample of descriptions from the best experiment using the Task 1 rubric to confirm improvement.

In [7]:
# Choose the best experiment result (change variable as needed)
best_results = exp2_results  # <-- update after reviewing comparison table

sample_idx = np.linspace(0, len(best_results) - 1, 5, dtype=int)
for i in sample_idx:
    row = best_results.iloc[i]
    print(f"--- [{i}] {row['product_name']} ({row['word_count']} words) ---")
    print(row["generated_description"])
    print()

--- [0] Apple iPhone 15 Pro (75 words) ---
The iPhone 15 Pro packs powerful performance and stunning visuals into a compact design. Experience lightning-fast speeds with the A17 Pro chip and enjoy incredibly smooth scrolling with the 120Hz ProMotion display.  Crafted with a durable titanium frame and Ceramic Shield glass, the iPhone 15 Pro is built to last. Plus, enjoy the convenience of USB-C fast charging. Backed by a one-year limited warranty, the iPhone 15 Pro is the ultimate smartphone for demanding users.

--- [12] DJI Mini 4 Pro Drone (66 words) ---
Capture stunning 4K/60fps footage with the DJI Mini 4 Pro, a compact drone weighing under 250g.  Omnidirectional obstacle sensing ensures safe and confident flight, while the lightweight composite plastic design makes it incredibly portable.  Averaging a 4.7/5 star rating, the Mini 4 Pro delivers professional-quality aerial photography and videography in a user-friendly package. Backed by a 1-year limited warranty, you can fly with p

## Summary

| Experiment | What changed | Result |
|------------|-------------|--------|
| Exp 1 | Few-shot prompt (SYSTEM_PROMPT_V2) | Length compliance improved from 98% to 100%. Mean word count dropped from 69.9 to 67.8, tighter around the target range. |
| Exp 2 | Lower temperature (0.3) with few-shot prompt | Compliance dropped slightly to 96% — 2 descriptions exceeded 90 words (Samsung at 99w, Fitbit at 92w). Mean rose to 69.2. |
| Exp 3 | Llama 3.1-8B with few-shot + temp 0.3 | 100% compliance again, lowest mean word count (65.1). But latency tripled to ~3.1s avg, with some calls hitting 5+ seconds. |

**Best configuration:** Experiment 1 — few-shot prompt with Gemma at default temperature (0.7). It achieved 100% length compliance with fast latency (~986ms avg). The few-shot example gave the model a concrete target without over-constraining it.

**What helped most:** The few-shot prompt. Adding a concrete example of a good description was the single biggest improvement — it took compliance from 98% to 100% and tightened the word count distribution.

**What didn't help:** Lowering temperature actually hurt slightly (2 descriptions went over 90 words at temp 0.3, vs. 0 at temp 0.7). The Llama model achieved the same 100% compliance but with 3x worse latency, making it a worse trade-off for generation.